# Land Cover Area Calculation with GEE

In [1]:
# =========================
# 0) Imports & GEE init
# =========================
import ee
import geopandas as gpd
import json

# ---- Config (edit these) ----
PROJECT_ID = "ee-2015893776"
AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"

# Export settings
EXPORT_FOLDER = None       # e.g., "GEE_LC_Outputs" ; None means Drive root
EXPORT_RASTERS = False     # True: export yearly clipped GeoTIFFs for QA
RASTER_EXPORT_MAXPIXELS = 1e13

# =========================
# 1) Initialize Earth Engine
# =========================
ee.Initialize(project=PROJECT_ID)

In [2]:
# =========================
# 2) Read AOI (shp) -> ee.Geometry
# =========================
gdf = gpd.read_file(AOI_SHP)

# ensure EPSG:4326
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

geojson = json.loads(gdf.to_json())
aoi_fc = ee.FeatureCollection(geojson)
geom = aoi_fc.geometry()   # union geometry for all features

In [3]:
# =========================
# 3) Dataset IDs
# =========================
# C3S LC LCCS (annual, ~300m)
LC_C3S = "projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS"

# ESRI 10m Annual Land Cover time series
LC_ESRI = "projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"

In [4]:
# =========================
# 4) Your reclass mapping (C3S LCCS -> 1..7)
# =========================
RECLASS_MAP_C3S = {
    # Forest -> 1
    50: 1, 60: 1, 61: 1, 62: 1, 70: 1, 71: 1, 72: 1, 80: 1, 81: 1, 82: 1, 90: 1, 100: 1,
    # Grassland -> 2
    110: 2, 120: 2, 121: 2, 122: 2, 130: 2, 140: 2, 150: 2, 151: 2, 152: 2, 153: 2,
    # Cropland -> 3
    10: 3, 11: 3, 12: 3, 20: 3, 30: 3, 40: 3,
    # Wetlands -> 4
    160: 4, 170: 4, 180: 4,
    # Artificial -> 5
    190: 5,
    # Other -> 6
    200: 6, 201: 6, 202: 6, 220: 6,
    # Water -> 7
    210: 7,
    # No data -> 0
    0: 0,
}
C3S_FROM = list(RECLASS_MAP_C3S.keys())
C3S_TO   = list(RECLASS_MAP_C3S.values())

In [5]:
# =========================
# 5) ESRI class mapping (ESRI codes -> your 1..7)
# =========================
# ESRI Global LULC 10m classes are typically:
# 1 Water, 2 Trees, 4 Flooded Vegetation, 5 Crops, 7 Built Area,
# 8 Bare Ground, 9 Snow/Ice, 10 Clouds, 11 Rangeland
RECLASS_MAP_ESRI = {
    2: 1,    # Trees -> Forest
    11: 2,   # Rangeland -> Grassland
    5: 3,    # Crops -> Cropland
    4: 4,    # Flooded vegetation -> Wetlands
    7: 5,    # Built area -> Artificial
    8: 6,    # Bare ground -> Other
    1: 7,    # Water -> Water
    10: 0,   # Clouds -> No data
    9: 6,    # Snow/Ice -> Other (if you prefer NoData, change to 0)
}
ESRI_FROM = list(RECLASS_MAP_ESRI.keys())
ESRI_TO   = list(RECLASS_MAP_ESRI.values())

In [6]:
# =========================
# 6) Helper: detect band names safely (only small getInfo)
# =========================
def get_first_band_name(ic: ee.ImageCollection, label: str) -> str:
    bnames = ic.first().bandNames().getInfo()
    if not bnames:
        raise ValueError(f"{label}: cannot detect band names (empty).")
    print(f"{label} bandNames =", bnames)
    return bnames[0]

# Build collections with AOI bounds filter (cheap, helps)
c3s_ic_all = ee.ImageCollection(LC_C3S).filterBounds(geom)
esri_ic_all = ee.ImageCollection(LC_ESRI).filterBounds(geom)

# Detect bands
C3S_BAND = None
# C3S often uses "lccs_class". If not present, fallback to first band.
c3s_bnames = c3s_ic_all.first().bandNames().getInfo()
print("C3S bandNames =", c3s_bnames)
if "lccs_class" in c3s_bnames:
    C3S_BAND = "lccs_class"
else:
    C3S_BAND = c3s_bnames[0]
    print(f"Warning: using C3S band '{C3S_BAND}' (not 'lccs_class').")

ESRI_BAND = get_first_band_name(esri_ic_all, "ESRI")

C3S bandNames = ['b1']
ESRI bandNames = ['b1']


In [37]:
# =========================
# 7) Core: area km2 by class (grouped pixelArea)
# =========================
def area_km2_by_class(class_img, geom, scale, tileScale=8):
    area_img = ee.Image.pixelArea().addBands(class_img.rename("cls"))
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
        geometry=geom,
        scale=scale,
        maxPixels=1e13,
        tileScale=tileScale
    )
    groups = ee.List(stats.get("groups"))

    def to_kv(d):
        d = ee.Dictionary(d)
        return ee.List([ee.Number(d.get("class")).format(),
                        ee.Number(d.get("sum")).divide(1e6)])
    return ee.Dictionary(groups.map(to_kv).flatten())


def dict_to_long_fc(area_dict: ee.Dictionary, year: int, source: str) -> ee.FeatureCollection:
    keys = area_dict.keys()
    def make_feat(k):
        k = ee.String(k)
        return ee.Feature(None, {
            "year": year,
            "source": source,
            "class": ee.Number.parse(k),
            "area_km2": area_dict.getNumber(k)
        })
    return ee.FeatureCollection(keys.map(make_feat))

In [36]:
# =========================
# 8) Reclass functions + alignment
# =========================
def reclass_c3s(img: ee.Image, geom: ee.Geometry) -> ee.Image:
    lc = img.select(C3S_BAND).clip(geom)
    return lc.remap(C3S_FROM, C3S_TO).rename("cls").toInt16()

def reclass_esri_to_300m(img, geom, target_proj):
    cls10 = img.select(ESRI_BAND).clip(geom).toInt16()

    # 关键修复：给输入影像一个可用的 default projection（用它自己第一景的投影）
    cls10 = cls10.setDefaultProjection(cls10.projection())

    # 如果你原来这里有 reduceResolution，就不会再报错
    cls300 = (cls10
              .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=4096)
              .reproject(crs=target_proj.crs(), scale=300))

    return cls300.remap(ESRI_FROM, ESRI_TO).rename("cls").toInt16()


In [9]:
# =========================
# 9) Export helpers
# =========================
def export_table_to_drive(fc: ee.FeatureCollection, desc: str):
    params = dict(
        collection=fc,
        description=desc,
        fileFormat="CSV"
    )
    if EXPORT_FOLDER:
        params["folder"] = EXPORT_FOLDER

    task = ee.batch.Export.table.toDrive(**params)
    task.start()
    print("Export started (CSV):", desc)

def export_image_to_drive(img: ee.Image, desc: str, scale: int):
    params = dict(
        image=img,
        description=desc,
        region=geom,
        scale=scale,
        maxPixels=RASTER_EXPORT_MAXPIXELS,
        fileFormat="GeoTIFF"
    )
    if EXPORT_FOLDER:
        params["folder"] = EXPORT_FOLDER

    task = ee.batch.Export.image.toDrive(**params)
    task.start()
    print("Export started (GeoTIFF):", desc)



In [11]:
# =========================
# 10) Main: C3S 2000-2015 area table (AOI clipped)
# =========================
import pandas as pd
c3s_ic = c3s_ic_all.filterDate("2000-01-01", "2015-12-31")

rows = []

# 先只看少量年份，避免并发聚合/429（确认无误后再改成 range(2000, 2016)）
for y in range(2000, 2016):  # <-- 先检查 2000-2002
    img_y = c3s_ic.filterDate(f"{y}-01-01", f"{y}-12-31").first()
    cls_y = reclass_c3s(img_y, geom)

    # 服务器端计算（reduceRegion），这里返回 ee.Dictionary
    area_dict = area_km2_by_class(cls_y, geom, scale=300)

    area_py = area_dict.getInfo()  # e.g. {"1": 123.4, "2": 567.8, ...}

    # 组装成长表
    for k, v in area_py.items():
        rows.append({
            "year": y,
            "source": "C3S_300m",
            "class": int(k),
            "area_km2": float(v)
        })

df_c3s = pd.DataFrame(rows).sort_values(["year", "class"]).reset_index(drop=True)
df_matrix = (
    df_c3s
    .pivot(index="year", columns="class", values="area_km2")
    .sort_index()
    .sort_index(axis=1)
)

df_matrix.to_csv( r"../../Part2_SustainableDevelopmentGoal_LandDegradation/Result/baseline_landcover.csv" )
df_matrix

class,1,2,3,4,5,6,7
year,,,,,,,
2000,178.695153,3366.838140,2548.698154,51.948366,2660.984945,62205.148258,313.726843
2001,178.734175,3342.819303,2546.087377,52.002975,2664.149391,62228.857660,313.388976
2002,179.266529,3343.485735,2546.335105,52.247643,2668.518767,62229.506193,306.679886
2003,179.789976,3343.268033,2545.847910,52.247643,2670.059185,62230.169602,304.657509
2004,181.675683,3343.595994,2547.380031,54.333756,2673.053904,62236.464282,289.536209
2005,183.572316,3333.118916,2544.614779,54.496660,2675.713020,62254.752822,279.771346
2006,183.528402,3338.455021,2542.636963,54.496660,2680.210491,62261.773184,264.939137
2007,183.528402,3338.878809,2541.822627,54.496660,2683.366331,62261.670314,262.276715
2008,186.095934,3344.995768,2541.737641,58.416771,2687.437592,62263.837561,243.518593


In [38]:
# =========================
# 11) Main: ESRI 2019-2024 reclass + align to 300m + area table
# =========================
import pandas as pd
target_proj = c3s_ic.first().select(C3S_BAND).projection()

esri_ic = esri_ic_all.filterDate("2018-01-01", "2025-12-31")

rows = []

for y in range(2018, 2026):
    ic_y = esri_ic.filterDate(f"{y}-01-01", f"{y}-12-31")

    # 可选：避免某年缺数据直接报错
    if ic_y.size().getInfo() == 0:
        print("Missing ESRI year:", y)
        continue

    img_y = ic_y.mosaic()

    # 关键：从某个 tile 里拿一个“真实的投影”，贴回 mosaic 影像
    proj10 = ee.Image(ic_y.first()).select(ESRI_BAND).projection()
    img_y = img_y.setDefaultProjection(proj10)

    cls300_y = reclass_esri_to_300m(img_y, geom, target_proj)


    area_dict = area_km2_by_class(cls300_y, geom, scale=300, tileScale=8)
    area_py = area_dict.getInfo()

    for k, v in area_py.items():
        rows.append({
            "year": y,
            "source": "ESRI_10m_to_300m",
            "class": int(k),
            "area_km2": float(v)
        })
    print(f'Processed ESRI year: {y}')

df_esri = pd.DataFrame(rows).sort_values(["year", "class"]).reset_index(drop=True)

# year × class 的宽表（缺失类别填 0）
df_esri_matrix = (
    df_esri
    .pivot(index="year", columns="class", values="area_km2")
    .reindex(columns=range(0, 8), fill_value=0)
    .sort_index()
)
df_esri_matrix.to_csv( r"../../Part2_SustainableDevelopmentGoal_LandDegradation/Result/Reporting_landcover.csv" )
df_esri_matrix

Processed ESRI year: 2018
Processed ESRI year: 2019
Processed ESRI year: 2020
Processed ESRI year: 2021
Processed ESRI year: 2022
Processed ESRI year: 2023
Processed ESRI year: 2024
Missing ESRI year: 2025


class,0,1,2,3,4,5,6,7
year,,,,,,,,
2018,0,4.145602,17069.737377,270.867465,16.918393,2629.698865,50913.081552,421.545909
2019,0,4.630645,17884.143346,292.287440,23.586187,2892.059222,49806.949905,422.338419
2020,0,5.054876,18858.339723,255.198431,29.627697,3205.643545,48553.007657,419.123235
2021,0,5.295094,19755.661132,246.175407,27.862547,3178.598587,47686.775788,425.626608
2022,0,3.792138,20132.795529,188.044714,22.436202,3582.291431,46985.482492,411.152658
2023,0,2.523687,20701.889032,186.668418,1.190613,3793.188894,46176.892995,463.641525
2024,0,4.063938,20801.116922,151.038815,1.110889,4227.067356,45665.980670,475.616573
